# MLflow Tracking + Model Registry

Instruments the day-ahead forecasting pipeline with MLflow. Every backtest becomes
a parent run with one nested run per fold, LightGBM is registered as `champion`,
and the LSTM-vs-LightGBM significance test is queried straight from MLflow rather
than recomputed.

**Prerequisite:** `pip install "mlflow>=2.9"` in this env.

Sections:
1. Setup — SQLite store and experiment
2. Data and features
3. Backtest models with logging
4. Train and register the production model
5. Load the registered model back and verify
6. Compare two models via MLflow queries (paired significance test)
7. Launch the MLflow UI

## 1. Setup

In [1]:
from pathlib import Path

EXPERIMENT_NAME = "electricity-forecasting"
REGISTERED_NAME = "electricity-forecaster"
STORE_DIR       = "mlflow_store"    # local SQLite + artifacts folder
DATA_PATH       = "data/Electricity Consumption 2015-2020.csv"

HORIZON  = 24
N_FOLDS  = 4
COUNTRY  = "TR"
RUN_LSTM = True    # logging works, but adds ~10 minutes

In [2]:
import os, sys, warnings
from pathlib import Path

# Auto-locate the repo root (same trick as run_pipeline.ipynb).
_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "src").is_dir() and (_p / "data").is_dir():
        os.chdir(_p)
        sys.path.insert(0, str(_p))
        print(f"Repo root: {_p}")
        break
else:
    raise RuntimeError("Could not find repo root")

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import mlflow

from src.data           import load_and_clean
from src.features       import make_features
from src.backtest       import backtest_model, make_splitter
from src.models         import make_ridge, make_random_forest, make_gbm
from src.mlflow_logging import (
    setup_tracking, log_backtest, register_production_model, load_registered,
)

print("MLflow version:", mlflow.__version__)

Repo root: C:\Users\ahmed\OneDrive\Desktop\Time series forecasting\electricity-consumption-forecasting-main
MLflow version: 3.16.1


In [3]:
tracking_uri = setup_tracking(EXPERIMENT_NAME, store_dir=STORE_DIR)
print("Tracking URI :", tracking_uri)
print("Experiment   :", EXPERIMENT_NAME)

Tracking URI : sqlite:///C:/Users/ahmed/OneDrive/Desktop/Time series forecasting/electricity-consumption-forecasting-main/mlflow_store/mlflow.db
Experiment   : electricity-forecasting


## 2. Data and features

In [4]:
series, audit = load_and_clean(DATA_PATH)
frame = make_features(series, min_lag=HORIZON, country=COUNTRY)
splitter = make_splitter(n_splits=N_FOLDS)
print(f"Frame: {frame.shape[0]:,} rows × {frame.shape[1] - 1} features")

Frame: 38,736 rows × 61 features


## 3. Backtest models with logging

Each `backtest_model` call is wrapped in `log_backtest`. Same numbers as
`run_pipeline.ipynb` — the only new thing is that everything gets written to
MLflow.

In [5]:
common_params = {
    "horizon_hours":    HORIZON,
    "n_folds":          N_FOLDS,
    "country_holidays": COUNTRY,
    "n_features":       frame.shape[1] - 1,
    "backtest_scheme":  "expanding, 90d test, 24h gap",
}

model_specs = {
    "Ridge":         (make_ridge,         {"model_kind": "linear"}),
    "Random Forest": (make_random_forest, {"model_kind": "tree_ensemble"}),
    "LightGBM":      (make_gbm,           {"model_kind": "gradient_boosting"}),
}

for name, (factory, extra) in model_specs.items():
    print(f"→ {name:<15s}", end=" ", flush=True)
    fold_results, importance, preds = backtest_model(
        frame, factory, name, splitter, collect_predictions=True,
    )
    run_id = log_backtest(
        model_name=name,
        fold_results=fold_results,
        params={**common_params, **extra},
        importance=importance,
        predictions=preds,
        tags={"kind": "backtest", "horizon": "day_ahead"},
    )
    print(f"logged {run_id[:8]}   mean MAE {fold_results['MAE'].mean():,.1f}")

→ Ridge           logged 93119c0b   mean MAE 1,127.6
→ Random Forest   logged 9d3e3267   mean MAE 833.7
→ LightGBM        logged 4967ade3   mean MAE 841.8


In [6]:
if RUN_LSTM:
    from src.lstm_backtest import backtest_lstm

    print("→ LSTM (5–10 min) ...", end=" ", flush=True)
    lstm_results, lstm_preds = backtest_lstm(series, frame, splitter)
    run_id = log_backtest(
        model_name="LSTM",
        fold_results=lstm_results.drop(columns=["epochs"], errors="ignore"),
        params={**common_params, "model_kind": "sequence_nn", "seq_len": 72},
        predictions=lstm_preds,
        tags={"kind": "backtest", "horizon": "day_ahead"},
    )
    print(f"logged {run_id[:8]}   mean MAE {lstm_results['MAE'].mean():,.1f}")
else:
    print("LSTM skipped (RUN_LSTM = False).")

→ LSTM (5–10 min) ... logged d8768787   mean MAE 792.3


## 4. Train and register the production model

Backtests prove the approach. The production model is trained on all available
data. LightGBM is the champion — within 1% of the LSTM's MAE at roughly 15× less
training cost.

In [7]:
import mlflow.sklearn

X, y = frame.drop(columns="y"), frame["y"]

with mlflow.start_run(run_name="LightGBM (production)"):
    mlflow.set_tags({"kind": "production", "purpose": "serving"})
    mlflow.log_params({**common_params, "trained_on": "all_available_data"})

    prod_model = make_gbm().fit(X, y)

    mlflow.sklearn.log_model(
        prod_model,
        artifact_path="model",
        input_example=X.head(3),
        registered_model_name=REGISTERED_NAME,
        serialization_format="pickle",
    )

    client  = mlflow.MlflowClient()
    latest  = max(int(v.version) for v in client.search_model_versions(f"name='{REGISTERED_NAME}'"))
    client.set_registered_model_alias(REGISTERED_NAME, "champion", str(latest))
    version = str(latest)

print(f"Registered {REGISTERED_NAME} v{version} with alias 'champion'")

2026/09/29 01:25:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/29 01:25:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered model 'electricity-forecaster' already exists. Creating a new version of this model...
Created version '3' of model 'electricity-forecaster'.


Registered electricity-forecaster v3 with alias 'champion'


## 5. Load the registered model back and verify

Loading by alias (not by version) means FastAPI never needs to know which
version is current — flip the alias to promote a new model.

In [8]:
loaded = load_registered(REGISTERED_NAME, alias="champion")

# Smoke test on the last 24 hours.
sample = X.tail(24)
preds  = loaded.predict(sample)

print(f"Loaded model : {type(loaded).__name__}")
print(f"Predictions on the last 24 h — min {preds.min():,.0f}"
      f", mean {preds.mean():,.0f}, max {preds.max():,.0f}")

Loaded model : LGBMRegressor
Predictions on the last 24 h — min 28,803, mean 35,305, max 39,146


## 6. LSTM vs LightGBM significance test — via MLflow queries

Pulls per-fold MAE from the two parent runs and re-runs the paired t-test the
README describes. Result is logged as its own "comparison" run so the metrics
live in the same experiment as the models they compare.

If `RUN_LSTM = False`, this falls back to LightGBM vs Random Forest so the
cell always produces something to look at.

In [9]:
from scipy import stats

client = mlflow.MlflowClient()
exp_id = client.get_experiment_by_name(EXPERIMENT_NAME).experiment_id


def fetch_fold_mae(model_name: str) -> list[float]:
    """Return per-fold MAE ordered by fold number for one model's runs."""
    parents = client.search_runs(
        exp_id,
        filter_string=(
            f"tags.mlflow.runName = '{model_name}' and tags.kind = 'backtest'"
        ),
        max_results=1,
    )
    if not parents:
        raise ValueError(f"No backtest run found for '{model_name}'")
    parent_id = parents[0].info.run_id
    children = client.search_runs(
        exp_id,
        filter_string=f"tags.mlflow.parentRunId = '{parent_id}'",
    )
    ordered = sorted(children, key=lambda r: int(r.data.params["fold"]))
    return [r.data.metrics["MAE"] for r in ordered]


model_a, model_b = ("LSTM", "LightGBM") if RUN_LSTM else ("LightGBM", "Random Forest")
a, b = fetch_fold_mae(model_a), fetch_fold_mae(model_b)

tstat, p = stats.ttest_rel(a, b)
diff = float(np.mean(np.array(a) - np.array(b)))

with mlflow.start_run(run_name=f"{model_a} vs {model_b}"):
    mlflow.set_tags(
        {"kind": "comparison", "test": "paired_t",
         "model_a": model_a, "model_b": model_b}
    )
    mlflow.log_metric("t_statistic",   float(tstat))
    mlflow.log_metric("p_value",       float(p))
    mlflow.log_metric("mean_diff_MAE", diff)

print(f"{model_a} vs {model_b}")
print(f"  {model_a:<15s} per-fold MAE: {[round(x, 1) for x in a]}")
print(f"  {model_b:<15s} per-fold MAE: {[round(x, 1) for x in b]}")
print(f"  t = {tstat:+.3f}   p = {p:.3f}")
print(f"  mean MAE difference ({model_a} - {model_b}): {diff:+.1f}")
print(f"  significant at 0.05? {'yes' if p < 0.05 else 'no'}")

LSTM vs LightGBM
  LSTM            per-fold MAE: [764.1, 535.9, 717.9, 1151.5]
  LightGBM        per-fold MAE: [819.7, 524.3, 738.5, 1284.6]
  t = -1.591   p = 0.210
  mean MAE difference (LSTM - LightGBM): -49.4
  significant at 0.05? no


## 7. Launch the MLflow UI

From a terminal at the repo root:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow_store/mlflow.db --port 5000
```

Then open http://localhost:5000. You should see:

- **Experiments → electricity-forecasting** — one parent run per model, expand
  each to see its four fold children.
- **Compare** — tick two or more parent runs, click Compare, and MLflow plots
  the metrics side by side.
- **Models → electricity-forecaster** — the production LightGBM with alias
  `champion` pointing at v1.

The whole state lives in `mlflow_store/`. Zip it, share it, or drop it into a
Docker volume later.